# NFL game-day lift near the 11 World Cup stadiums

**Question this answers:** when 65–80k people go to a stadium, how much extra card spend shows up
at food / fuel / lodging / venue shops nearby, how far from the stadium does it reach, and on which
days around the game does it show up?

**Why this is measurable despite the fake numbers:** the organisers put *multiplicative* noise on
every dollar figure. A ratio (game day ÷ normal day) for the same shops cancels a multiplicative
factor, so ratios survive even though levels don't.

**What you get:** a table of lift ratios by distance ring (0–2, 2–5, 5–10, 10–25 km) and by day
offset (−3 … +3 around the game). 1.00 means "same as a normal day of that weekday",
1.30 means 30% more.

Inputs: `spend-patterns-rice.parquet` in your Drive cache (already there from `explore_raw.ipynb`),
plus `stadiums.csv` and `nfl_home_games.csv` from `data/worldcup/` in the repo — put them in the
same cache folder or upload them when asked.

In [ ]:
# ============================================================
# STEP 0 — mount Drive, find the inputs
# ============================================================
import os, json, io
import numpy as np, pandas as pd

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/ricehack_cache"
except ImportError:
    CACHE = os.path.expanduser("~/ricehack_cache")

MUST_HAVE = {"copa_america_2024.csv": {"date","stadium","market"}, "stadiums.csv": {"stadium","market","lat","lon"},
             "nfl_home_games.csv": {"date","stadium","market"}}
def need(name):
    p = f"{CACHE}/{name}"
    while True:
        if os.path.exists(p) and os.path.getsize(p) > 0:
            try:
                t = pd.read_csv(p)
                if MUST_HAVE[name].issubset(t.columns): return t
            except Exception: pass
            os.remove(p); print(f"{name} in the cache was empty or the wrong file — removed it")
        print(f"upload {name} from data/worldcup/ in the repo (any filename is fine)")
        up = files.upload()
        open(p, "wb").write(list(up.values())[0])

stadiums = need("stadiums.csv")
games    = need("nfl_home_games.csv")
games["date"] = pd.to_datetime(games["date"])
print(len(stadiums), "stadiums,", len(games), "home games", games.date.min().date(), "to", games.date.max().date())


In [ ]:
# ============================================================
# STEP 1 — load only the shops within 25 km of a stadium
# ============================================================
RINGS = [(0, 2), (2, 5), (5, 10), (10, 25)]     # km from the stadium
MAX_KM = RINGS[-1][1]

EFW_NAICS3 = {"722","445","447","721","711","712","713","562","221","311","312","424","485","481","488"}
def layer_of(n3):
    if n3 in {"722","445","311","312","424"}: return "Food"
    if n3 in {"447","221"}:                   return "Energy"
    if n3 in {"721","562"}:                   return "Water"
    if n3 in {"711","712","713"}:             return "Venue"
    return "Other_EFW"

COLS = ["PLACEKEY","MARKET","LATITUDE","LONGITUDE","NAICS_CODE",
        "SPEND_DATE_RANGE_START","SPEND_BY_DAY","RAW_TOTAL_SPEND"]
df = pd.read_parquet(f"{CACHE}/spend-patterns-rice.parquet", columns=COLS)
print(f"{len(df):,} shop-months loaded")

df["n3"] = df.NAICS_CODE.astype(str).str[:3]
df = df[df.n3.isin(EFW_NAICS3)].copy()
df["layer"] = df.n3.map(layer_of)
df["LATITUDE"]  = pd.to_numeric(df.LATITUDE,  errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df.LONGITUDE, errors="coerce")
df = df.dropna(subset=["LATITUDE","LONGITUDE"])
print(f"{len(df):,} EFW shop-months with coordinates")

# distance from each shop to its own market's stadium
st = stadiums.set_index("market")
lat0 = df.MARKET.map(st.lat); lon0 = df.MARKET.map(st.lon)
df["km"] = np.sqrt(((df.LATITUDE - lat0) * 111) ** 2
                   + ((df.LONGITUDE - lon0) * 111 * np.cos(np.radians(lat0))) ** 2)
df = df[df.km <= MAX_KM].copy()
df["ring"] = pd.cut(df.km, [r[0] for r in RINGS] + [MAX_KM], labels=[f"{a}-{b} km" for a, b in RINGS], include_lowest=True)
df["stadium"] = df.MARKET.map(st.stadium)
print(f"{len(df):,} shop-months within {MAX_KM} km of a stadium\n")
print(df.groupby(["stadium","ring"], observed=True).PLACEKEY.nunique().unstack().fillna(0).astype(int).to_string())
print("\n^ distinct shops per ring. A ring with only a handful of shops will give noisy ratios.")


In [ ]:
# ============================================================
# STEP 2 — unpack SPEND_BY_DAY into daily totals per stadium x ring x layer
# ============================================================
def as_vals(v):
    if isinstance(v, dict): return [v[k] for k in sorted(v)]
    if isinstance(v, (list, np.ndarray)): return list(v)
    if isinstance(v, str):
        try: p = json.loads(v)
        except Exception: return []
        return [p[k] for k in sorted(p)] if isinstance(p, dict) else list(p)
    return []

s_arr = df.SPEND_BY_DAY.map(as_vals)
n_days = s_arr.map(len).values
keep = n_days > 0
df, s_arr, n_days = df[keep], s_arr[keep], n_days[keep]

flat = np.concatenate(s_arr.values).astype("float64")
dates = np.concatenate([pd.date_range(s, periods=n, freq="D").values
                        for s, n in zip(pd.to_datetime(df.SPEND_DATE_RANGE_START), n_days)])
daily = pd.DataFrame({
    "stadium": np.repeat(df.stadium.values, n_days),
    "ring":    np.repeat(df.ring.astype(str).values, n_days),
    "layer":   np.repeat(df.layer.values, n_days),
    "date":    dates,
    "spend":   flat,
}).groupby(["stadium","ring","layer","date"], as_index=False).spend.sum()

# also an "all EFW" layer, which is the headline
allrow = daily.groupby(["stadium","ring","date"], as_index=False).spend.sum(); allrow["layer"] = "All"
daily = pd.concat([daily, allrow], ignore_index=True)
daily.to_parquet(f"{CACHE}/stadium_ring_daily.parquet", index=False)
print(f"{len(daily):,} stadium x ring x layer x day rows, {daily.date.min().date()} to {daily.date.max().date()}")


In [ ]:
# ============================================================
# STEP 3 — lift = game-window day  /  same weekday on non-game weeks
# ============================================================
OFFSETS  = range(-3, 4)      # days around the game
CTRL_WKS = [-4, -3, -2, -1, 1, 2, 3, 4]   # same weekday, these many weeks away

games["with_crowd"] = ~((games.season == 2020))   # 2020 = empty / capped stadiums = built-in control
# every date within +-3 days of ANY home game at that stadium is "contaminated" and never used as control
contam = set()
for r in games.itertuples():
    for k in range(-3, 4): contam.add((r.stadium, (r.date + pd.Timedelta(days=k)).normalize()))

wide = daily.pivot_table(index=["stadium","ring","layer"], columns="date", values="spend", fill_value=0.0)
rows = []
for g in games.itertuples():
    for k in OFFSETS:
        d = (g.date + pd.Timedelta(days=k)).normalize()
        if d not in wide.columns: continue
        ctrl_dates = [d + pd.Timedelta(weeks=w) for w in CTRL_WKS]
        ctrl_dates = [c for c in ctrl_dates if c in wide.columns and (g.stadium, c) not in contam]
        if len(ctrl_dates) < 3: continue
        sub = wide.loc[g.stadium]
        obs  = sub[d]
        ctrl = sub[ctrl_dates].median(axis=1)
        ok = ctrl > 0
        for (ring, layer), o, c in zip(sub.index[ok], obs[ok], ctrl[ok]):
            rows.append(dict(stadium=g.stadium, date=g.date, season=g.season, with_crowd=g.with_crowd,
                             weekday=g.weekday, offset=k, ring=ring, layer=layer, obs=o, ctrl=c, lift=o / c))
L = pd.DataFrame(rows)
L.to_csv(f"{CACHE}/nfl_lift_long.csv", index=False)
print(f"{len(L):,} game x offset x ring x layer ratios; {L.groupby(['stadium','date']).ngroups} game-days used")


In [ ]:
# ============================================================
# STEP 4 — the tables you actually read
# ============================================================
pd.set_option("display.width", 200)
def q(x, p): return x.quantile(p)

def table(sub, title):
    t = sub.groupby(["ring","offset"]).lift.agg(median="median", p25=lambda x: q(x,.25), p75=lambda x: q(x,.75), n="count")
    print(f"\n=== {title} ===")
    print("median lift (1.00 = normal day of that weekday). Rows = distance ring, columns = days from the game")
    print(t["median"].unstack("offset").round(2).to_string())
    print("\ninter-quartile range width (bigger = less trustworthy)")
    print((t["p75"] - t["p25"]).unstack("offset").round(2).to_string())

crowd = L[L.with_crowd & (L.layer == "All")]
table(crowd, "A. All EFW shops, games WITH a crowd (2021-2024)")
table(L[~L.with_crowd & (L.layer == "All")], "B. Same, but 2020 games (empty / capped stadiums) — should sit near 1.00 everywhere")

# three buckets, which is what the forecast will actually use
b = crowd.copy()
b["bucket"] = pd.cut(b.offset, [-4, -2, 1, 3], labels=["before (-3,-2)", "around (-1,0,+1)", "after (+2,+3)"])
bt = (b.groupby(["ring","bucket"], observed=True).apply(lambda x: x.obs.sum() / x.ctrl.sum()).unstack("bucket").round(3))
print("\n=== C. Same games, pooled dollars: sum(observed) / sum(control), by ring and bucket ===")
print(bt.to_string())

print("\n=== D. By type of shop, ring 0-2 km, 'around' bucket ===")
d = L[L.with_crowd & (L.ring == "0-2 km") & L.offset.between(-1, 1)]
print(d.groupby("layer").apply(lambda x: pd.Series({"pooled_lift": x.obs.sum()/x.ctrl.sum(), "median_lift": x.lift.median(), "n": len(x)})).round(3).to_string())

print("\n=== E. Per stadium, ring 0-2 km, 'around' bucket — which stadiums have a usable signal ===")
e = crowd[(crowd.ring == "0-2 km") & crowd.offset.between(-1, 1)]
print(e.groupby("stadium").apply(lambda x: pd.Series({"pooled_lift": x.obs.sum()/x.ctrl.sum(), "median_lift": x.lift.median(), "games": x.date.nunique()})).round(3).to_string())

summary = bt.reset_index().melt(id_vars="ring", var_name="bucket", value_name="lift")
summary.to_csv(f"{CACHE}/nfl_lift_by_ring_bucket.csv", index=False)
print(f"\nsaved {CACHE}/nfl_lift_by_ring_bucket.csv — this is the file the forecast will read")


## How to read what comes back

**Table A is the answer.** Look at the `0-2 km` row first. Offset `0` is game day. Because
`SPEND_BY_DAY` is stamped on the day the card payment *settled*, a Sunday game's money mostly
appears on offset `+1` (Monday), so read `0` and `+1` together as "the game". If both sit near
1.00 even at 0–2 km, the data can't see the crowd at all and we stop here.

**Offsets −3, −2 and +2, +3 answer the three-day question.** If they are near 1.00, an NFL crowd
is a day-trip crowd and the arrival/departure spill is not something we can claim from this data.
If they sit clearly above 1.00 (say 1.10+) at 0–2 km, there is a real multi-day bump and its size
is right there. Remember an NFL crowd is mostly local, so this is a *floor* for a World Cup crowd,
not the whole story.

**Read across the rings** to see how fast the effect fades with distance. That fade curve, from
Table C, is the "scaling factor by radius" the forecast needs. `10-25 km` should be close to
1.00; if it isn't, the control weeks are picking up something citywide (holidays, weather) and
the ratios are less trustworthy.

**Table B is the sanity check.** 2020 games had empty or capped stadiums. Every cell should sit
near 1.00. If Table B shows the same bumps as Table A, the method is measuring the football
*season*, not the crowd, and nothing above can be trusted.

**The IQR-width table** says how much games disagree with each other. Widths under about 0.3
around a median of 1.2 are fine; widths of 1.0+ mean the ring is too thin to trust (check the
shops-per-ring print in Step 1).

**Table D** tells you whether the lift is food and drink (expect yes), fuel (probably a little),
lodging (the out-of-town tell — if lodging moves at all, people are staying over) or venues.

**Table E** shows which stadiums the data can actually see. Downtown stadiums (Lumen, Mercedes-Benz,
Lincoln Financial) should show more shops and cleaner ratios than car-park stadiums (Gillette,
Arrowhead, AT&T), where the nearest shops may be 5+ km away.
